# Sentinel-2 Time Series Download (Kaggle Edition)

Downloads Sentinel-2 L2A stacks via Microsoft Planetary Computer (no signup) for the
smallholder irrigation dataset. Designed to run end-to-end on a free Kaggle notebook:

- Uses `/kaggle/working/` as output (auto-saved to the notebook's downloadable artifacts)
- Stays under Kaggle's 20 GB output limit by downloading ~1,150 sites (all KL test labels +
  stratified train sample). For a full reproduction you'd run multiple notebooks.
- Resumes automatically if the runtime is killed and you re-run the notebook.

## How to use
1. Sign in at https://kaggle.com (instant, free).
2. Create a new Notebook → upload this `.ipynb` file.
3. In the right-hand panel, make sure 'Internet' is **On** (Settings → Internet → On).
4. Click **Save Version → Save & Run All (Commit)**. Close the tab. Come back in ~4 hours.
5. When done, open the saved version → **Output** tab → download the `s2_stacks.zip`.

## 1. Install extra packages

Kaggle already has pandas, numpy, rasterio, requests. We add the STAC client + MPC URL-signer.

In [ ]:
!pip install -q pystac-client planetary-computer

## 2. Imports and config

In [ ]:
import os, sys, json, math, logging, shutil
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import datetime, timedelta

import numpy as np
import pandas as pd
import rasterio
import requests
from rasterio.enums import Resampling
from rasterio.warp import transform as warp_transform
from rasterio.windows import from_bounds

import pystac_client
import planetary_computer

logging.basicConfig(level=logging.INFO,
                    format='%(asctime)s [%(levelname)s] %(message)s',
                    force=True)

# Kaggle paths
OUT_ROOT = '/kaggle/working/data/features/sentinel2'
os.makedirs(OUT_ROOT, exist_ok=True)

# Constants matching the project's GEE-based downloader
S2_BANDS = ['B2','B3','B4','B5','B6','B7','B8','B8A','B11','B12']
MPC_KEY  = {'B2':'B02','B3':'B03','B4':'B04','B5':'B05','B6':'B06',
            'B7':'B07','B8':'B08','B8A':'B8A','B11':'B11','B12':'B12'}
GOOD_SCL = {4, 5, 6, 7, 11}    # vegetation/soil/water/unclassified/snow
AOI_HALF_M = 500               # ~1 km AOI half-side
TARGET_SIZE = 100              # 100x100 px at 10m resolution

print('Output root:', OUT_ROOT)

## 3. Fetch the label CSV from GitHub

Pulls `latest_irrigation_table.csv` directly from the public repo so we don't need a Kaggle Dataset.

In [ ]:
CSV_URL = ('https://raw.githubusercontent.com/AIscend-Research/'
           'smallholder-irrigation-dataset/main/'
           'data/labels/labeled_surveys/random_sample/latest_irrigation_table.csv')
CSV_LOCAL = '/kaggle/working/latest_irrigation_table.csv'

r = requests.get(CSV_URL, timeout=60)
r.raise_for_status()
with open(CSV_LOCAL, 'wb') as f:
    f.write(r.content)

df = pd.read_csv(CSV_LOCAL)
print('rows:', len(df))
print('unique site+date:', df.groupby(['site_id','year','month','day']).ngroups)
df.head(3)

## 4. Build the site list (≤1,150 sites to fit Kaggle's 20 GB output)

Strategy:
- **Always include every site where KL labeled it** (KL is the project's test set per CLAUDE.md)
- Then fill remaining budget with a stratified sample of the rest (by year) for the train set.

In [ ]:
BUDGET = 1150              # ~ 16 MB/site → ~18 GB output, under Kaggle's 20 GB cap

# 1) Dedupe to one row per unique (site, date), so each file is downloaded once
uniq = df.drop_duplicates(subset=['site_id','year','month','day']).copy()
uniq['ymd'] = uniq.apply(lambda r: f"{int(r.year):04d}{int(r.month):02d}{int(r.day):02d}", axis=1)

# 2) Flag any (site,date) where KL is one of the labelers → must include
kl_keys = set(df[df['operator_initials']=='KL']
              .apply(lambda r: (r['site_id'], r['year'], r['month'], r['day']), axis=1))
uniq['has_kl'] = uniq.apply(lambda r: (r['site_id'], r['year'], r['month'], r['day']) in kl_keys, axis=1)

kl_set = uniq[uniq['has_kl']]
rest   = uniq[~uniq['has_kl']]
print(f'KL test images (must-include): {len(kl_set)}')
print(f'Remaining candidates: {len(rest)}')

# 3) Stratified sample from rest by year to fill the budget
remaining_budget = max(0, BUDGET - len(kl_set))
if remaining_budget > 0 and len(rest) > 0:
    per_year = rest.groupby('year', group_keys=False).apply(
        lambda g: g.sample(
            n=min(len(g), max(1, int(round(remaining_budget * len(g) / len(rest))))),
            random_state=42)
    )
    # Adjust if rounding overshot
    if len(per_year) > remaining_budget:
        per_year = per_year.sample(n=remaining_budget, random_state=42)
else:
    per_year = rest.iloc[:0]

selected = pd.concat([kl_set, per_year]).reset_index(drop=True)
print(f'Total selected: {len(selected)}')
print('Year distribution:')
print(selected['year'].astype(int).value_counts().sort_index())

## 5. STAC download helpers (inlined from `src/features/download_sentinel2_stac.py`)

Same logic as the project's STAC downloader — just inlined here so this notebook is self-contained.

In [ ]:
_catalog = None
def get_catalog():
    global _catalog
    if _catalog is None:
        _catalog = pystac_client.Client.open(
            'https://planetarycomputer.microsoft.com/api/stac/v1',
            modifier=planetary_computer.sign_inplace,
        )
    return _catalog

def aoi_bounds_utm(lat, lon, dst_crs):
    xs, ys = warp_transform('EPSG:4326', dst_crs, [lon], [lat])
    cx, cy = xs[0], ys[0]
    return (cx-AOI_HALF_M, cy-AOI_HALF_M, cx+AOI_HALF_M, cy+AOI_HALF_M)

def search_best_item(lat, lon, start, end):
    cat = get_catalog()
    items = list(cat.search(
        collections=['sentinel-2-l2a'],
        intersects={'type':'Point','coordinates':[lon, lat]},
        datetime=f'{start}/{end}',
    ).items())
    if not items: return None
    items.sort(key=lambda it: it.properties.get('eo:cloud_cover') or math.inf)
    return items[0]

def read_band(href, bounds_utm, resampling):
    with rasterio.open(href) as src:
        win = from_bounds(*bounds_utm, transform=src.transform)
        arr = src.read(1, window=win, out_shape=(TARGET_SIZE, TARGET_SIZE),
                       resampling=resampling, boundless=True, fill_value=0)
        win_t = src.window_transform(win)
        sx, sy = win.width/TARGET_SIZE, win.height/TARGET_SIZE
        return arr.astype(np.uint16), win_t * rasterio.Affine.scale(sx, sy), src.crs

def write_stack(path, stack, transform, crs):
    bands, h, w = stack.shape
    with rasterio.open(path, 'w', driver='GTiff', height=h, width=w, count=bands,
                       dtype='uint16', crs=crs, transform=transform, nodata=0,
                       compress='deflate') as dst:
        dst.write(stack)

def export_one_scene(lat, lon, start, end, file_name, out_dir):
    os.makedirs(out_dir, exist_ok=True)
    item = search_best_item(lat, lon, start, end)
    if item is None: return False
    with rasterio.open(item.assets['B02'].href) as s: scene_crs = s.crs
    bounds = aoi_bounds_utm(lat, lon, scene_crs)
    try:
        bands, tfm, crs = [], None, None
        for b in S2_BANDS:
            arr, t, c = read_band(item.assets[MPC_KEY[b]].href, bounds, Resampling.bilinear)
            bands.append(arr)
            if tfm is None: tfm, crs = t, c
        scl, _, _ = read_band(item.assets['SCL'].href, bounds, Resampling.nearest)
    except Exception as e:
        logging.error(f'read failed {item.id}: {e}')
        return False
    unmasked = np.stack(bands, axis=0)
    good = np.isin(scl, list(GOOD_SCL))
    masked = unmasked * good[np.newaxis,:,:].astype(np.uint16)
    write_stack(os.path.join(out_dir, f'{file_name}.tif'), unmasked, tfm, crs)
    write_stack(os.path.join(out_dir, f'{file_name}_masked.tif'), masked, tfm, crs)
    return True

def retrieve_time_series(file_id, lat, lon, date, out_dir,
                          start_month=1, num_windows=36, timestep=10,
                          window_buffer=3):
    step = timedelta(days=timestep)
    n_buf = num_windows + window_buffer*2
    start_d = datetime(date.year, start_month, 1) - step*window_buffer
    wins = [(start_d + i*step, start_d + (i+1)*step) for i in range(n_buf)]

    tmp = os.path.join(out_dir, '_tmp', file_id)
    os.makedirs(tmp, exist_ok=True)

    def paths(s, e):
        nm = f's2_{lat:.2f}_{lon:.2f}_{s.strftime("%Y-%m-%d")}_{e.strftime("%Y-%m-%d")}'
        return os.path.join(tmp, nm+'.tif'), os.path.join(tmp, nm+'_masked.tif'), s.strftime('%Y-%m-%d'), e.strftime('%Y-%m-%d')

    with ThreadPoolExecutor(max_workers=10) as ex:
        futs = []
        for s, e in wins:
            u, m, ss, es = paths(s, e)
            if not (os.path.exists(u) and os.path.exists(m)):
                nm = f's2_{lat:.2f}_{lon:.2f}_{ss}_{es}'
                futs.append(ex.submit(export_one_scene, lat, lon, ss, es, nm, tmp))
        for f in as_completed(futs):
            try: f.result()
            except Exception as e: logging.error(f'export err: {e}')

    empty = np.zeros((len(S2_BANDS), TARGET_SIZE, TARGET_SIZE), dtype=np.uint16)
    im_u, im_m, meta_w = [], [], []
    tcrs = ttf = None
    for s, e in wins:
        u, m, ss, es = paths(s, e)
        if os.path.exists(u) and os.path.exists(m):
            try:
                with rasterio.open(u) as src:
                    arr_u = src.read()
                    if tcrs is None: tcrs, ttf = src.crs, src.transform
                with rasterio.open(m) as src:
                    arr_m = src.read()
                im_u.append(arr_u); im_m.append(arr_m)
                frac = float(((arr_m==0).any(axis=0)).sum()/(TARGET_SIZE*TARGET_SIZE))
                meta_w.append({'date_range':[ss,es],'file_exists':True,'masked_fraction':frac})
            except Exception as e:
                logging.error(f'read fail: {e}')
                im_u.append(empty.copy()); im_m.append(empty.copy())
                meta_w.append({'date_range':[ss,es],'file_exists':False,'masked_fraction':1.0})
        else:
            im_u.append(empty.copy()); im_m.append(empty.copy())
            meta_w.append({'date_range':[ss,es],'file_exists':False,'masked_fraction':1.0})

    if tcrs is None:
        shutil.rmtree(tmp, ignore_errors=True)
        raise RuntimeError(f'No valid images for {file_id}')

    su = np.stack(im_u, axis=0); sm = np.stack(im_m, axis=0)
    T, B, H, W = su.shape
    ru = su.transpose(1,0,2,3).reshape(T*B, H, W)
    rm = sm.transpose(1,0,2,3).reshape(T*B, H, W)
    for path, arr in [(f'{out_dir}/{file_id}_stack.tif', ru),
                      (f'{out_dir}/{file_id}_stack_masked.tif', rm)]:
        with rasterio.open(path, 'w', driver='GTiff', height=H, width=W, count=T*B,
                           dtype=su.dtype, crs=tcrs, transform=ttf, nodata=0,
                           compress='deflate') as dst:
            dst.write(arr)

    meta = {'file_id': file_id,'lat': float(lat),'lon': float(lon),
            'year': int(date.year),'collection':'L2A','bands': S2_BANDS,
            'shape': list(sm.shape),'target_size': TARGET_SIZE,
            'num_windows': num_windows,'num_windows_buffered': n_buf,
            'timestep_days': timestep,'start_month_unbuffered': start_month,
            'window_buffer': window_buffer,'windows': meta_w}
    with open(f'{out_dir}/{file_id}_metadata.json','w') as f:
        json.dump(meta, f, indent=2)
    shutil.rmtree(tmp, ignore_errors=True)

## 6. Run the download (resume-safe)

Site-level parallelism set to 3. Re-running the cell after a session kill picks up where it left off.

In [ ]:
VERSION = 'kaggle_run'
out_dir = os.path.join(OUT_ROOT, VERSION)
os.makedirs(out_dir, exist_ok=True)

MAX_CONCURRENT_SITES = 3

def process_row(row):
    lat, lon = row['y'], row['x']
    date = datetime(int(row['year']), int(row['month']), int(row['day']))
    sid = str(row['site_id']).replace('id_','')
    file_id = f"{sid}_{date.year}.{date.month:02d}.{date.day:02d}"
    if os.path.exists(f'{out_dir}/{file_id}_stack.tif'):
        return 'skip', file_id, None
    try:
        retrieve_time_series(file_id, lat, lon, date, out_dir)
        return 'done', file_id, None
    except Exception as e:
        return 'fail', file_id, str(e)

rows = [r for _, r in selected.iterrows()]
total = len(rows)
done = skipped = failed = 0
logging.info(f'Starting: {total} sites, {MAX_CONCURRENT_SITES} concurrent')

with ThreadPoolExecutor(max_workers=MAX_CONCURRENT_SITES) as ex:
    futs = {ex.submit(process_row, r): r for r in rows}
    for f in as_completed(futs):
        status, fid, err = f.result()
        n = done + skipped + failed + 1
        if   status == 'done': done += 1;    logging.info(f'[{n}/{total}] done {fid}')
        elif status == 'skip': skipped += 1; logging.info(f'[{n}/{total}] skip {fid}')
        else:                  failed += 1;  logging.error(f'[{n}/{total}] FAIL {fid}: {err}')

logging.info(f'Finished: {done} done, {skipped} skipped, {failed} failed')

## 7. Zip everything for easy download

Kaggle saves anything in `/kaggle/working/` to your notebook's Output tab. The zip just makes it one click.

In [ ]:
import subprocess
ZIP_PATH = '/kaggle/working/s2_stacks.zip'
if os.path.exists(ZIP_PATH):
    os.remove(ZIP_PATH)
# -j flag would flatten paths; we keep the directory structure intact
subprocess.run(['zip', '-r', '-q', ZIP_PATH, out_dir, CSV_LOCAL])
size_mb = os.path.getsize(ZIP_PATH) / 1024 / 1024
print(f'Zip ready: {ZIP_PATH}  ({size_mb:.0f} MB)')
print('Download from the Output tab of this notebook after Save & Run All finishes.')

## What to do after this runs

1. Download `s2_stacks.zip` from this notebook's Output tab to your laptop.
2. Unzip into your local project at `data/features/sentinel2/kaggle_run/`.
3. Continue with Step 2 of Phase 1:
   ```bash
   python src/features/create_label_band.py --download_dir data/features/sentinel2 --version kaggle_run
   ```
4. Then Step 3 (build training dataset via the modeling pipeline).